# 3. Advanced Data Tools

**Goal:** Empower the agent to query enterprise data warehouses (like Snowflake) to answer factual business questions.

**Key Concept:**
We use the **Model Context Protocol (MCP)** to connect our agent to a DataRobot deployment acting as a secure "Data Tool." This setup enables the agent to dynamically generate queries and retrieve live datasets—transforming it from a simple chatbot into a data analyst capable of answering questions like *"What distinct bakeries are we tracking supplies for?"*

In [1]:
from dotenv import load_dotenv
from pprint import pprint
import datarobot as dr
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP

load_dotenv()
dr_client = dr.Client()

MCP_DEPLOYMENT_ID = "692eee8f6362220c3956af6e"  # <-- Replace with your actual deployment ID

server = MCPServerStreamableHTTP(
    f"{dr_client.endpoint}/deployments/{MCP_DEPLOYMENT_ID}/directAccess/mcp",
    headers={
        "Authorization": f"Bearer {dr_client.token}",
        "x-datarobot-api-token": dr_client.token,
    },
    timeout=60.0,
)

MODEL_NAME = "azure/gpt-5-2025-08-07"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token, base_url=dr_client.endpoint + "/genai/llmgw"
    ),
)

system_prompt = """
Use the "Snowflake Agent Demo" Data Source to answer data questions about our supplies.

"""
agent = Agent(model=model, toolsets=[server], system_prompt=system_prompt)

async with server:
    response = await agent.run("What distinct bakeries are we tracking supplies for ?")
    pprint(response.output)


('Here are the distinct bakeries we’re tracking supplies for (from '
 'SANDBOX.LUVRO.BUTTER_ONGOING_SUPPLY):\n'
 '\n'
 '- Desert Rose Bakery\n'
 '- Empire State Bakery Co.\n'
 '- Golden Coast Bakehouse\n'
 '- Great Lakes Artisan Bread\n'
 '- Lone Star Artisan Bakery\n'
 '- New England Heritage Baking\n'
 '- Northern Plains Bread House\n'
 '- Pacific Northwest Grain Co.\n'
 '- Peach Tree Baking Works\n'
 '- Rocky Mountain Fresh Bakery')


In [4]:
async with server:
    response = await agent.run("How many unique hub_name are in dataset 6972a732b304ef35e947921a")
    pprint(response.output)

'There are 4 unique hub_name values in dataset 6972a732b304ef35e947921a.'
